# RSNA Knee Abnormality Detection — First Submission
## Six-slot Target-Attention ResNet18 inference

This notebook is designed for **Kaggle Code Competition submission**.

It:

1. uses only test MRI + frozen model weights;
2. does no training;
3. requires Internet OFF;
4. creates `/kaggle/working/submission.csv`;
5. preserves the exact competition column order.

The hidden scoring environment replaces the tiny example test data with the real test set.

### Required inputs

```text
RSNA competition data
phase4b2b1_final_sixslot_targetattn_resnet18_v1.pt
```

The checkpoint should be uploaded to a Kaggle Dataset or Kaggle Model and attached to this notebook.

In [ ]:
from pathlib import Path
from functools import lru_cache
import os, re, time, warnings

import numpy as np
import pandas as pd
import cv2
import pydicom

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from torchvision.models import resnet18
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

COMP_DIR = Path(
    "/kaggle/input/competitions/rsna-knee-abnormality-detection"
)

OUT_DIR = Path("/kaggle/working")

TARGETS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]

SLOTS = [
    "SAG_FS", "SAG_NONFS",
    "COR_FS", "COR_NONFS",
    "AX_FS", "AX_NONFS",
]
SLOT_TO_ID = {s: i for i, s in enumerate(SLOTS)}

CROP_MM = 130.0
IMAGE_SIZE = 224
INFERENCE_ANCHORS = 8
SLICE_BAND = (0.12, 0.88)

BATCH_SIZE = 1
NUM_WORKERS = 2

print("Device:", DEVICE)

## 1. Find the final checkpoint

In [ ]:
checkpoint_hits = []

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    if root.exists():
        checkpoint_hits.extend(
            root.rglob(
                "phase4b2b1_final_sixslot_targetattn_resnet18_v1.pt"
            )
        )

if len(checkpoint_hits) != 1:
    print("Checkpoint matches:")
    for p in checkpoint_hits:
        print(" -", p)

    raise RuntimeError(
        "Attach exactly one Phase 4B-2B1 final checkpoint."
    )

CHECKPOINT_PATH = checkpoint_hits[0]

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False,
)

print("Checkpoint:", CHECKPOINT_PATH)
print("Experiment:", checkpoint.get("experiment_version"))
print("Config:", checkpoint.get("model_config"))

assert checkpoint["model_config"]["architecture"] == (
    "SixSlotTargetAttention_ResNet18_2p5D"
)

## 2. Build the test six-slot plan

In [ ]:
test = pd.read_csv(
    COMP_DIR / "test.csv"
)

test_series = pd.read_csv(
    COMP_DIR / "test_series.csv"
)

sample = pd.read_csv(
    COMP_DIR / "sample_submission.csv"
)

test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)
test_series["StudyInstanceUID"] = test_series["StudyInstanceUID"].astype(str)
test_series["SeriesInstanceUID"] = test_series["SeriesInstanceUID"].astype(str)

TEST_SERIES_ROOT = COMP_DIR / "test_series"

def as_binary(v):
    if pd.isna(v):
        return 0

    if isinstance(v, (int, float, np.integer, np.floating)):
        return int(float(v) > 0)

    return int(
        str(v).strip().lower()
        in {"1", "true", "yes", "y"}
    )

def slot_name(plane, fat):
    plane = str(plane)
    fat = int(fat)

    prefix = {
        "Sagittal": "SAG",
        "Coronal": "COR",
        "Axial": "AX",
    }.get(plane)

    if prefix is None:
        return None

    return (
        f"{prefix}_FS"
        if fat == 1
        else f"{prefix}_NONFS"
    )

test_series["fat_bin"] = (
    test_series["Fat_Suppression"].map(as_binary)
)

test_series["fluid_bin"] = (
    test_series["Fluid_Sensitive"].map(as_binary)
)

test_series["slot"] = [
    slot_name(p, f)
    for p, f in zip(
        test_series["Anatomical_Plane"],
        test_series["fat_bin"],
    )
]

def series_path_for(row):
    return (
        TEST_SERIES_ROOT
        / str(row.StudyInstanceUID)
        / str(row.SeriesInstanceUID)
    )

file_counts = []

for row in tqdm(
    test_series.itertuples(index=False),
    total=len(test_series),
    desc="count series files",
):
    p = series_path_for(row)

    try:
        n = sum(
            1 for x in p.iterdir()
            if x.is_file()
        )
    except Exception:
        n = 0

    file_counts.append(n)

test_series["n_files"] = file_counts

usable = test_series.loc[
    test_series["slot"].notna()
].copy()

usable = usable.sort_values(
    [
        "StudyInstanceUID",
        "slot",
        "n_files",
        "fluid_bin",
        "SeriesInstanceUID",
    ],
    ascending=[
        True, True, False, False, True
    ],
)

test_slot_plan = (
    usable
    .groupby(
        ["StudyInstanceUID", "slot"],
        as_index=False,
    )
    .first()
)

test_slot_plan["series_path"] = [
    str(
        TEST_SERIES_ROOT
        / uid
        / sid
    )
    for uid, sid in zip(
        test_slot_plan["StudyInstanceUID"],
        test_slot_plan["SeriesInstanceUID"],
    )
]

print("Test studies:", len(test))
print("Selected test slot-series:", len(test_slot_plan))
print(
    test_slot_plan.groupby("slot")["StudyInstanceUID"]
    .nunique()
)

## 3. DICOM ordering + 130-mm crop + 2.5D anchors

In [ ]:
def header_record(path):
    try:
        ds = pydicom.dcmread(
            str(path),
            stop_before_pixels=True,
            force=True,
            specific_tags=[
                "ImagePositionPatient",
                "ImageOrientationPatient",
                "InstanceNumber",
            ],
        )

        ipp = getattr(
            ds,
            "ImagePositionPatient",
            None,
        )

        iop = getattr(
            ds,
            "ImageOrientationPatient",
            None,
        )

        inst = getattr(
            ds,
            "InstanceNumber",
            None,
        )

        spatial = None

        if (
            ipp is not None
            and iop is not None
            and len(ipp) >= 3
            and len(iop) >= 6
        ):
            row = np.asarray(
                iop[:3],
                dtype=np.float64,
            )

            col = np.asarray(
                iop[3:6],
                dtype=np.float64,
            )

            normal = np.cross(
                row,
                col,
            )

            norm = np.linalg.norm(
                normal
            )

            if norm > 1e-8:
                normal /= norm

                spatial = float(
                    np.dot(
                        np.asarray(
                            ipp[:3],
                            dtype=np.float64,
                        ),
                        normal,
                    )
                )

        inst_value = (
            float(inst)
            if inst is not None
            else np.nan
        )

        return (
            str(path),
            spatial,
            inst_value,
        )

    except Exception:
        return (
            str(path),
            None,
            np.nan,
        )


@lru_cache(maxsize=12000)
def ordered_series_files(series_path):
    files = sorted([
        p for p in Path(series_path).iterdir()
        if p.is_file()
    ])

    records = [
        header_record(p)
        for p in files
    ]

    n_spatial = sum(
        r[1] is not None
        for r in records
    )

    if (
        n_spatial
        >= max(
            2,
            int(0.8 * len(records)),
        )
    ):
        records = sorted(
            records,
            key=lambda r: (
                float("inf")
                if r[1] is None
                else r[1],
                r[0],
            ),
        )

    elif (
        np.isfinite(
            [r[2] for r in records]
        ).sum()
        >= max(
            2,
            int(0.8 * len(records)),
        )
    ):
        records = sorted(
            records,
            key=lambda r: (
                float("inf")
                if not np.isfinite(r[2])
                else r[2],
                r[0],
            ),
        )

    return tuple(
        r[0]
        for r in records
    )


def normalize_pixels(
    arr,
    photometric="MONOCHROME2",
):
    x = np.asarray(
        arr,
        dtype=np.float32,
    )

    finite = np.isfinite(x)

    if not finite.any():
        return np.zeros_like(
            x,
            dtype=np.float32,
        )

    vals = x[finite]

    lo, hi = np.percentile(
        vals,
        [1.0, 99.0],
    )

    if hi <= lo + 1e-6:
        lo = float(vals.min())
        hi = float(vals.max())

    if hi <= lo + 1e-6:
        return np.zeros_like(
            x,
            dtype=np.float32,
        )

    x = np.clip(
        (x - lo) / (hi - lo),
        0.0,
        1.0,
    )

    if str(photometric).upper() == "MONOCHROME1":
        x = 1.0 - x

    x[~finite] = 0.0

    return x.astype(np.float32)


def crop_130mm(
    x,
    spacing_row,
    spacing_col,
):
    h, w = x.shape[:2]

    sr = float(spacing_row)
    sc = float(spacing_col)

    crop_h = (
        int(round(CROP_MM / sr))
        if np.isfinite(sr) and sr > 0
        else h
    )

    crop_w = (
        int(round(CROP_MM / sc))
        if np.isfinite(sc) and sc > 0
        else w
    )

    crop_h = max(
        8,
        min(crop_h, h),
    )

    crop_w = max(
        8,
        min(crop_w, w),
    )

    y0 = max(
        (h - crop_h) // 2,
        0,
    )

    x0 = max(
        (w - crop_w) // 2,
        0,
    )

    return x[
        y0:y0 + crop_h,
        x0:x0 + crop_w,
    ]


def decode_slice(path):
    try:
        ds = pydicom.dcmread(
            str(path),
            force=True,
        )

        arr = ds.pixel_array

        if arr.ndim > 2:
            arr = np.asarray(
                arr
            )[0]

        spacing = getattr(
            ds,
            "PixelSpacing",
            [np.nan, np.nan],
        )

        sr = (
            float(spacing[0])
            if len(spacing) >= 1
            else np.nan
        )

        sc = (
            float(spacing[1])
            if len(spacing) >= 2
            else np.nan
        )

        x = normalize_pixels(
            arr,
            getattr(
                ds,
                "PhotometricInterpretation",
                "MONOCHROME2",
            ),
        )

        x = crop_130mm(
            x,
            sr,
            sc,
        )

        x = cv2.resize(
            x,
            (IMAGE_SIZE, IMAGE_SIZE),
            interpolation=cv2.INTER_AREA,
        )

        return x.astype(
            np.float32
        )

    except Exception:
        return None


def anchor_indices(n, k):
    if n <= 0:
        return np.zeros(
            k,
            dtype=np.int64,
        )

    lo = int(
        np.floor(
            SLICE_BAND[0] * (n - 1)
        )
    )

    hi = int(
        np.ceil(
            SLICE_BAND[1] * (n - 1)
        )
    )

    return np.clip(
        np.rint(
            np.linspace(
                lo,
                hi,
                k,
            )
        ),
        0,
        n - 1,
    ).astype(np.int64)


def load_25d_series(
    series_path,
    k,
):
    files = ordered_series_files(
        str(series_path)
    )

    n = len(files)

    out = np.zeros(
        (
            k,
            3,
            IMAGE_SIZE,
            IMAGE_SIZE,
        ),
        dtype=np.float32,
    )

    valid = np.zeros(
        k,
        dtype=np.float32,
    )

    if n == 0:
        return out, valid

    anchors = anchor_indices(
        n,
        k,
    )

    for j, i in enumerate(
        anchors
    ):
        ids = [
            max(i - 1, 0),
            i,
            min(i + 1, n - 1),
        ]

        imgs = [
            decode_slice(files[q])
            for q in ids
        ]

        good = [
            x for x in imgs
            if x is not None
        ]

        if not good:
            continue

        fallback = good[
            len(good) // 2
        ]

        imgs = [
            x if x is not None
            else fallback
            for x in imgs
        ]

        out[j] = np.stack(
            imgs,
            axis=0,
        )

        valid[j] = 1.0

    return out, valid

## 4. Model architecture

In [ ]:
class SixSlotTargetAttention(nn.Module):

    def __init__(
        self,
        n_targets=12,
        n_slots=6,
        embed_dim=512,
        n_heads=8,
        dropout=0.2,
    ):
        super().__init__()

        self.backbone = resnet18(
            weights=None
        )

        self.backbone.fc = nn.Identity()

        self.slice_score = nn.Sequential(
            nn.Linear(
                embed_dim,
                128,
            ),
            nn.Tanh(),
            nn.Linear(
                128,
                1,
            ),
        )

        self.slot_embedding = nn.Embedding(
            n_slots,
            embed_dim,
        )

        self.target_queries = nn.Parameter(
            torch.randn(
                n_targets,
                embed_dim,
            ) * 0.02
        )

        self.cross_attention = nn.MultiheadAttention(
            embed_dim,
            n_heads,
            dropout=dropout,
            batch_first=True,
        )

        self.norm1 = nn.LayerNorm(
            embed_dim
        )

        self.ffn = nn.Sequential(
            nn.Linear(
                embed_dim,
                embed_dim * 2,
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),
            nn.Linear(
                embed_dim * 2,
                embed_dim,
            ),
        )

        self.norm2 = nn.LayerNorm(
            embed_dim
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.target_head_weight = nn.Parameter(
            torch.randn(
                n_targets,
                embed_dim,
            ) * 0.02
        )

        self.target_head_bias = nn.Parameter(
            torch.zeros(
                n_targets
            )
        )

    def forward(
        self,
        images,
        slice_mask,
        slot_mask,
    ):
        B, S, K, C, H, W = (
            images.shape
        )

        x = images.reshape(
            B * S * K,
            C,
            H,
            W,
        )

        feat = self.backbone(
            x
        )

        feat = feat.reshape(
            B,
            S,
            K,
            -1,
        )

        scores = (
            self.slice_score(
                feat
            ).squeeze(-1)
        )

        mask = (
            slice_mask.float()
        )

        scores = (
            scores
            - scores.amax(
                dim=-1,
                keepdim=True,
            )
        )

        exp_scores = (
            torch.exp(scores)
            * mask
        )

        attention = (
            exp_scores
            / (
                exp_scores.sum(
                    dim=-1,
                    keepdim=True,
                )
                + 1e-8
            )
        )

        slot_tokens = (
            feat
            * attention.unsqueeze(-1)
        ).sum(dim=2)

        slot_ids = torch.arange(
            S,
            device=images.device,
        )

        slot_tokens = (
            slot_tokens
            + self.slot_embedding(
                slot_ids
            )[None]
        )

        queries = (
            self.target_queries[
                None
            ].expand(
                B,
                -1,
                -1,
            )
        )

        attended, _ = (
            self.cross_attention(
                queries,
                slot_tokens,
                slot_tokens,
                key_padding_mask=(
                    slot_mask <= 0
                ),
                need_weights=False,
            )
        )

        h = self.norm1(
            queries
            + self.dropout(
                attended
            )
        )

        h = self.norm2(
            h
            + self.dropout(
                self.ffn(h)
            )
        )

        logits = (
            h
            * self.target_head_weight[
                None
            ]
        ).sum(
            dim=-1
        )

        return (
            logits
            + self.target_head_bias[
                None
            ]
        )


model = SixSlotTargetAttention(
    dropout=float(
        checkpoint[
            "model_config"
        ].get(
            "dropout",
            0.2,
        )
    )
)

missing, unexpected = (
    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ],
        strict=True,
    )
)

model.to(DEVICE)
model.eval()

print("✅ Checkpoint loaded strictly")

## 5. Test dataset and inference

In [ ]:
slot_rows_by_uid = {
    uid: {
        r.slot: r
        for r in d.itertuples(
            index=False
        )
    }
    for uid, d in test_slot_plan.groupby(
        "StudyInstanceUID"
    )
}


class TestStudyDataset(Dataset):

    def __init__(
        self,
        uids,
    ):
        self.uids = list(
            map(
                str,
                uids,
            )
        )

    def __len__(self):
        return len(
            self.uids
        )

    def __getitem__(
        self,
        idx,
    ):
        uid = self.uids[
            idx
        ]

        images = np.zeros(
            (
                len(SLOTS),
                INFERENCE_ANCHORS,
                3,
                IMAGE_SIZE,
                IMAGE_SIZE,
            ),
            dtype=np.float32,
        )

        slice_mask = np.zeros(
            (
                len(SLOTS),
                INFERENCE_ANCHORS,
            ),
            dtype=np.float32,
        )

        slot_mask = np.zeros(
            len(SLOTS),
            dtype=np.float32,
        )

        slot_dict = (
            slot_rows_by_uid.get(
                uid,
                {},
            )
        )

        for slot, sid in SLOT_TO_ID.items():

            if slot not in slot_dict:
                continue

            arr, valid = load_25d_series(
                slot_dict[
                    slot
                ].series_path,
                INFERENCE_ANCHORS,
            )

            if valid.sum() <= 0:
                continue

            images[sid] = arr
            slice_mask[sid] = valid
            slot_mask[sid] = 1.0

        if slot_mask.sum() <= 0:
            raise RuntimeError(
                f"No usable MRI slots for test study {uid}"
            )

        return {
            "StudyInstanceUID": uid,
            "images": torch.from_numpy(
                images
            ),
            "slice_mask": torch.from_numpy(
                slice_mask
            ),
            "slot_mask": torch.from_numpy(
                slot_mask
            ),
        }


test_loader = DataLoader(
    TestStudyDataset(
        test["StudyInstanceUID"]
        .astype(str)
        .tolist()
    ),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=(
        NUM_WORKERS > 0
    ),
)

uids = []
probs = []

start = time.time()

with torch.inference_mode():

    for batch in tqdm(
        test_loader,
        desc="test inference",
    ):
        images = batch[
            "images"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        slice_mask = batch[
            "slice_mask"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        slot_mask = batch[
            "slot_mask"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(
                DEVICE.type
                == "cuda"
            ),
        ):
            logits = model(
                images,
                slice_mask,
                slot_mask,
            )

        p = torch.sigmoid(
            logits
        ).float().cpu().numpy()

        uids.extend(
            list(
                batch[
                    "StudyInstanceUID"
                ]
            )
        )

        probs.append(p)

probs = np.concatenate(
    probs,
    axis=0,
)

print(
    "Inference minutes:",
    (time.time() - start) / 60,
)

## 6. Create the required `submission.csv`

In [ ]:
pred = pd.DataFrame(
    probs,
    columns=TARGETS,
)

pred.insert(
    0,
    "StudyInstanceUID",
    uids,
)

submission = (
    sample[
        ["StudyInstanceUID"]
    ]
    .astype(
        {
            "StudyInstanceUID":
                str
        }
    )
    .merge(
        pred,
        on="StudyInstanceUID",
        how="left",
        validate="one_to_one",
    )
)

expected_columns = [
    "StudyInstanceUID"
] + TARGETS

assert list(
    submission.columns
) == expected_columns

assert len(
    submission
) == len(
    sample
)

assert submission[
    TARGETS
].notna().all().all()

assert (
    submission[
        TARGETS
    ].to_numpy()
    >= 0
).all()

assert (
    submission[
        TARGETS
    ].to_numpy()
    <= 1
).all()

SUBMISSION_PATH = (
    OUT_DIR
    / "submission.csv"
)

submission.to_csv(
    SUBMISSION_PATH,
    index=False,
)

print("✅ submission.csv created")
print("Shape:", submission.shape)
display(submission.head())
print("Path:", SUBMISSION_PATH)

# Kaggle submission checklist

Before committing this notebook:

```text
GPU accelerator: ON
Internet: OFF
competition input: attached
final model checkpoint: attached
```

Then:

1. **Save Version / Commit**
2. choose **Save & Run All**
3. wait for the committed run to finish
4. verify `submission.csv` exists in the notebook output
5. click **Submit to Competition**

Do not enable training in this notebook.